# The Physics of Meaning: Text Embeddings and Semantic Search

In traditional software, we search for data using **Keywords** (SQL `LIKE` or Ctrl+F). But if a user searches for "feline" and your document says "cat," a keyword search fails. 

As an Architect, you solve this using **Embeddings**. 

An Embedding is a high-dimensional vector (a list of numbers) that represents the "essence" of a piece of text. By converting words into vectors, we can use **Geometry** to find related ideas. In this space, "Cat" and "Kitten" are mathematically close to each other, while "Cat" and "Nuclear Physics" are light-years apart.

In [ ]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
client = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    """Converts text into a 1,536-dimensional vector."""
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

print("📐 Vector Space Environment Ready!")
get_embedding("Apple")

## 1. Peering into the Vector

Let's convert a simple word into an embedding. Notice how many numbers are generated for just one word.

In [3]:
word = "Architect"
vector = get_embedding(word)

print(f"Word: {word}")
print(f"Vector Length (Dimensions): {len(vector)}")
print(f"First 5 numbers: {vector[:5]}")
print("\n💡 These 1,536 numbers represent the 'coordinates' of this word in OpenAI's map of human language.")

Word: Architect
Vector Length (Dimensions): 1536
First 5 numbers: [0.0228424072265625, -0.01117706298828125, 0.03179931640625, -0.020904541015625, -0.0113525390625]

💡 These 1,536 numbers represent the 'coordinates' of this word in OpenAI's map of human language.


## 2. Calculating Meaning (Cosine Similarity)

In vector physics, the **Cosine Similarity** tells us the angle between two vectors. 
- **1.0:** The ideas are identical.
- **0.0:** The ideas are completely unrelated.
- **-1.0:** The ideas are opposites.

In [4]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def compare_concepts(a, b):
    v1 = get_embedding(a)
    v2 = get_embedding(b)
    score = cosine_similarity(v1, v2)
    print(f"Similarity [{a}] vs [{b}]: {score:.4f}")

print("--- Concept Battlefield ---")
compare_concepts("Cat", "Kitten")
compare_concepts("Cat", "Dog")
compare_concepts("Cat", "Artificial Intelligence")
compare_concepts("King - Man + Woman", "Queen") # The classic 'Word Algebra' test

--- Concept Battlefield ---
Similarity [Cat] vs [Kitten]: 0.5537
Similarity [Cat] vs [Dog]: 0.5730
Similarity [Cat] vs [Artificial Intelligence]: 0.1608
Similarity [King - Man + Woman] vs [Queen]: 0.4062


## 3. Building a Semantic Search Engine

Now we apply this to a practical use case. We have a small database of sentences. We want to find the one that matches the *intent* of the user's query, even if the words are different.

In [6]:
database = [
    "The library is a quiet place to study.",
    "The stock market crashed yesterday due to inflation.",
    "Golden Retrievers are known for being friendly pets.",
    "Quantum computing uses qubits to perform calculations.",
    "Pizza is a popular Italian dish with cheese and tomato."
]

# 1. Pre-calculate embeddings for our database (The 'Indexing' phase)
print("Indexing database...")
db_vectors = [get_embedding(text) for text in database]

def semantic_search(query):
    # 2. Embed the user query
    query_vec = get_embedding(query)
    
    # 3. Compare query against every item in the database
    scores = [cosine_similarity(query_vec, db_vec) for db_vec in db_vectors]
    
    # 4. Find the best match
    best_idx = np.argmax(scores)
    return database[best_idx], scores[best_idx]

user_query = "I want to know, is it right time to invest?"
match, score = semantic_search(user_query)

display(Markdown(f"**User Query:** {user_query}"))
display(Markdown(f"**Top Match:** {match} (Score: {score:.4f})"))

Indexing database...


**User Query:** I want to know, is it right time to invest?

**Top Match:** The stock market crashed yesterday due to inflation. (Score: 0.2368)